In [1]:
import pandas as pd
import pyterrier as pt
import os
from ir_measures import *
import math

In [2]:
ohsumed_docs = pd.read_pickle("/nfs/primary/sas_reranker/ohsumed_docs_w_t5base_sensitivity.pkl")
queries = pd.read_pickle("/nfs/primary/graph_adaptive_reranking/queries.pkl")
qrels = pd.read_pickle("/nfs/primary/graph_adaptive_reranking/qrels.pkl")

In [3]:
def _sens_docs(qrels, run):
    if "sensitivity" in run.columns:
        run = run.drop(columns = ["sensitivity"])
    merged = pd.merge(run, ohsumed_docs, left_on = "doc_id", right_on = "docno")
    return merged.sensitivity.sum()
    
import ir_measures
sens_docs = ir_measures.define_byquery(
    _sens_docs, 
    name="sens_docs")

def calculate_csdcg_10(qrels, run):
    if "sensitivity" in run.columns:
        run = run.drop(columns = ["sensitivity"])
    merged = pd.merge(run, qrels, on = "doc_id", how = "left").fillna(0).head(10)
    merged = pd.merge(merged, ohsumed_docs, left_on = "doc_id", right_on = "docno")
    # print(merged[["relevance", "sensitivity"]])
    csdcg = 0
    for rank, doc in enumerate(merged.iterrows()):
        relevance = int(doc[1].relevance)
        sensitivity = int(doc[1]["sensitivity"])
        up = (2 ** relevance) - 1
        low = math.log(rank + 2)
        penalty = 12 if sensitivity == 1 else 0
        csdcg += ((up / low) - penalty)
    return csdcg

def _ncsdcg_cut_10(qrels, run):    
    # Calculate query csdcg
    query_csdcg = calculate_csdcg_10(qrels, run)
    
    # Calculate best csdcg
    merged_qrels = pd.merge(qrels, ohsumed_docs, left_on = "doc_id", right_on = "docno").drop(columns = ["title", "abstract", "text", "t5_sensitivity", "docno"])
    best_qrels =  merged_qrels.sort_values(by=['sensitivity', 'relevance'], ascending=[True, False])
    best_qrels = best_qrels.drop(columns = ["relevance", "sensitivity"])
    best_csdcg = calculate_csdcg_10(qrels, best_qrels)
    
    # Calculate worst csdcg
    worst_qrels =  merged_qrels.sort_values(by=['sensitivity', 'relevance'], ascending=[False, True])
    worst_qrels = worst_qrels.drop(columns = ["relevance", "sensitivity"])
    worst_csdcg = calculate_csdcg_10(qrels, worst_qrels)
    
    if query_csdcg <= worst_csdcg:
        query_csdcg = 0
        
    normalised_csdcg = (query_csdcg - worst_csdcg) / (best_csdcg - worst_csdcg)

    assert normalised_csdcg >= 0
    assert normalised_csdcg <= 1
    
    return normalised_csdcg
    
import ir_measures
nCSDCG_cut_10 = ir_measures.define_byquery(
    _ncsdcg_cut_10, 
    name="nCSDCG_cut_10")

def _ns_recall(qrels, run):
    combined = pd.merge(run, qrels, on = ["doc_id"])
    combined = pd.merge(combined, ohsumed_docs, left_on = "doc_id", right_on = "docno")
    combined.sensitivity = combined.sensitivity.astype(int)
    combined.relevance = combined.relevance.astype(int)
    num_rel_ns = combined[(combined['sensitivity'] < 1) & (combined['relevance'] > 1)].shape[0]
    qrels_w_sensitivity = pd.merge(qrels, ohsumed_docs, left_on = "doc_id", right_on = "docno")
    total = qrels_w_sensitivity[(qrels_w_sensitivity['sensitivity'] < 1) & (qrels_w_sensitivity['relevance'] > 1)].shape[0]
    assert num_rel_ns <= total
    if total == 0:
        return 0
    return num_rel_ns / total
    
import ir_measures
NSR = ir_measures.define_byquery(
    _ns_recall, 
    name="NSR")

def calculate_csdcg_20(qrels, run):
    if "sensitivity" in run.columns:
        run = run.drop(columns = ["sensitivity"])
    merged = pd.merge(run, qrels, on = "doc_id", how = "left").fillna(0).head(20)
    merged = pd.merge(merged, ohsumed_docs, left_on = "doc_id", right_on = "docno")
    # print(merged.columns)
    # print(merged[["relevance","sensitivity"]])
    csdcg = 0
    for rank, doc in enumerate(merged.iterrows()):
        relevance = int(doc[1].relevance)
        sensitivity = int(doc[1]["sensitivity"])
        up = (2 ** relevance) - 1
        low = math.log(rank + 2)
        penalty = 21 if sensitivity == 1 else 0
        csdcg += ((up / low) - penalty)
    return csdcg

def _ncsdcg_cut_20(qrels, run):    
    # Calculate query csdcg
    query_csdcg = calculate_csdcg_20(qrels, run)
    
    # Calculate best csdcg
    merged_qrels = pd.merge(qrels, ohsumed_docs, left_on = "doc_id", right_on = "docno").drop(columns = ["title", "abstract", "text", "t5_sensitivity", "docno"])
    best_qrels =  merged_qrels.sort_values(by=['sensitivity', 'relevance'], ascending=[True, False])
    best_qrels = best_qrels.drop(columns = ["relevance", "sensitivity"])
    best_csdcg = calculate_csdcg_20(qrels, best_qrels)
    
    # Calculate worst csdcg
    worst_qrels =  merged_qrels.sort_values(by=['sensitivity', 'relevance'], ascending=[False, True])
    worst_qrels = worst_qrels.drop(columns = ["relevance", "sensitivity"])
    worst_csdcg = calculate_csdcg_20(qrels, worst_qrels)

    normalised_csdcg = (query_csdcg - worst_csdcg) / (best_csdcg - worst_csdcg)
    return normalised_csdcg
    
import ir_measures
nCSDCG_cut_20 = ir_measures.define_byquery(
    _ncsdcg_cut_20, 
    name="nCSDCG_cut_20")

In [16]:
cat_one = []
cat_one_names = []
cat_two = []
cat_two_names = []
cat_three = []
cat_three_names = []

for file in os.listdir("./"):
    if not file.endswith(".csv") and not file.endswith(".ipynb") and not file.endswith("checkpoints") and not os.path.isdir(file):
        if ">>" not in file:
            cat_one.append(pt.io.read_results(file))
            cat_one_names.append(file)
        if ">>" in file and "monot5_relevance" in file and "filter" not in file:
            cat_two.append(pt.io.read_results(file))
            cat_two_names.append(file)
        if ">>" in file and "monot5_sans" in file and "filter" not in file:
            cat_three.append(pt.io.read_results(file))
            cat_three_names.append(file)

In [51]:
def run_significance_testing(runs, names, baseline):
    exp = pt.Experiment(
        runs,
        queries,
        qrels,
        eval_metrics=[nDCG@10, sens_docs@10, nCSDCG_cut_10, nDCG@20, sens_docs@20, nCSDCG_cut_20, sens_docs@100, NSR@100],
        names = names,
        baseline = baseline,
        correction = "b"
    )
    metrics = ["nDCG@10", "sens_docs@10", "nCSDCG_cut_10", "nDCG@20", "sens_docs@20", "nCSDCG_cut_20", "NSR@100", "sens_docs@100"]
    metrics_order = [f"{x} reject" for x in metrics]
    return exp[metrics_order]

In [55]:
run_significance_testing(cat_one, cat_one_names, 2)

,nDCG@10 reject,sens_docs@10 reject,nCSDCG_cut_10 reject,nDCG@20 reject,sens_docs@20 reject,nCSDCG_cut_20 reject,NSR@100 reject,sens_docs@100 reject
0,True,True,True,True,True,True,False,True
1,False,True,True,False,True,True,False,True
2,False,False,False,False,False,False,False,False
3,True,True,False,True,True,True,True,True


In [56]:
run_significance_testing(cat_two, cat_two_names, 2)

,nDCG@10 reject,sens_docs@10 reject,nCSDCG_cut_10 reject,nDCG@20 reject,sens_docs@20 reject,nCSDCG_cut_20 reject,NSR@100 reject,sens_docs@100 reject
0,False,True,True,False,True,False,False,True
1,False,False,False,False,False,False,False,True
2,False,False,False,False,False,False,False,False
3,True,True,True,True,True,True,True,True


In [57]:
run_significance_testing(cat_three, cat_three_names, 2)

,nDCG@10 reject,sens_docs@10 reject,nCSDCG_cut_10 reject,nDCG@20 reject,sens_docs@20 reject,nCSDCG_cut_20 reject,NSR@100 reject,sens_docs@100 reject
0,True,False,False,True,False,False,False,True
1,False,False,False,False,False,False,False,True
2,False,False,False,False,False,False,False,False
3,False,True,True,True,True,True,True,True


In [66]:
cat_one = []
cat_one_names = []
cat_two = []
cat_two_names = []
cat_three = []
cat_three_names = []

for file in os.listdir("./"):
    if not file.endswith(".csv") and not file.endswith(".ipynb") and not file.endswith("checkpoints"):
        if len(file.split(">>")) == 2:
            cat_one.append(pt.io.read_results(file))
            cat_one_names.append(file)
        if ">>" in file and "monot5_relevance" in file and "filter" in file:
            cat_two.append(pt.io.read_results(file))
            cat_two_names.append(file)
        if ">>" in file and "monot5_sans" in file and "filter" in file:
            cat_three.append(pt.io.read_results(file))
            cat_three_names.append(file)

In [77]:
run_significance_testing(cat_one, cat_one_names, 2)

,nDCG@10 reject,sens_docs@10 reject,nCSDCG_cut_10 reject,nDCG@20 reject,sens_docs@20 reject,nCSDCG_cut_20 reject,NSR@100 reject,sens_docs@100 reject
0,True,True,True,True,True,False,False,True
1,False,False,True,False,False,False,True,True
2,False,False,False,False,False,False,False,False
3,True,True,False,True,True,True,True,True


In [78]:
run_significance_testing(cat_two, cat_two_names, 2)

,nDCG@10 reject,sens_docs@10 reject,nCSDCG_cut_10 reject,nDCG@20 reject,sens_docs@20 reject,nCSDCG_cut_20 reject,NSR@100 reject,sens_docs@100 reject
0,False,True,True,False,True,False,False,True
1,False,False,False,False,False,False,True,True
2,False,False,False,False,False,False,False,False
3,True,True,True,True,True,True,True,True


In [79]:
run_significance_testing(cat_three, cat_three_names, 2)

,nDCG@10 reject,sens_docs@10 reject,nCSDCG_cut_10 reject,nDCG@20 reject,sens_docs@20 reject,nCSDCG_cut_20 reject,NSR@100 reject,sens_docs@100 reject
0,True,False,False,True,False,False,False,True
1,False,False,False,False,False,False,True,True
2,False,False,False,False,False,False,False,False
3,False,True,False,True,True,True,True,True
